In [2]:
import os 
import pandas as pd 
import numpy as np

In [9]:
df= pd.read_csv('/home/rajesh/work/ornella/unique_combination_cleaned.csv')

In [10]:
df.shape

(47696, 17)

In [11]:
df.isna().sum()

Unnamed: 0            0
Date Collected        0
Date Received         0
CowID                 0
Date Sent             0
Date Approved         0
ClientID              0
ProjectNo             0
Test                  0
Result                0
S-N Value             0
DIM                   0
DSLB                  0
combined_cow          0
date_collected        0
breeding_date         0
Unnamed: 16       47693
dtype: int64

In [12]:
# To see the total count of NaT values:
print(f"Total NaT values: {df['breeding_date'].isna().sum()}")

# To view all rows where breeding_date is NaT:
nat_rows = df[df['breeding_date'].isna()]

# Display the first few rows of the NaT records
print(nat_rows)


Total NaT values: 0
Empty DataFrame
Columns: [Unnamed: 0, Date Collected, Date Received, CowID, Date Sent, Date Approved, ClientID, ProjectNo, Test, Result, S-N Value, DIM, DSLB, combined_cow, date_collected, breeding_date, Unnamed: 16]
Index: []


In [13]:
df.columns

Index(['Unnamed: 0', 'Date Collected', 'Date Received', 'CowID', 'Date Sent',
       'Date Approved', 'ClientID', 'ProjectNo', 'Test', 'Result', 'S-N Value',
       'DIM', 'DSLB', 'combined_cow', 'date_collected', 'breeding_date',
       'Unnamed: 16'],
      dtype='object')

In [14]:
# 1. Strip the extra spaces, then 2. Convert to datetime
df["breeding_date"] = pd.to_datetime(df["breeding_date"].astype(str).str.strip(), format='mixed')
df['year']= df['breeding_date'].dt.year
df['month']= df['breeding_date'].dt.month
df['day']= df['breeding_date'].dt.day
df['month_day']= df['breeding_date'].dt.strftime('%m-%d')
# df = df.sort_values(["combined_cow", "breeding_date"])
# #Compute difference:
# df["prev_breed_date"] = df.groupby("combined_cow")["breeding_date"].shift()
# df["breed_diff_days"] = (df["breeding_date"] - df["prev_breed_date"]).dt.days
# #Flag abnormal:
# df["abnormal_breeding_interval"] = df["breed_diff_days"].between(1,2)
# #Round down:
# df.loc[df["abnormal_breeding_interval"], "breeding_date"] = df["prev_breed_date"]


In [ ]:
temp=[]
for combined_cow, data_cow in df.groupby(['combined_cow', 'year','month']):
    data_cow['date_diff']= data_cow['day'].diff()
    is_new_event = (data_cow['date_diff'] >= 2).fillna(True)
    data_cow['check_new_event']= is_new_event
    data_cow['cluster'] = is_new_event.cumsum()
    unique_cluster= len(data_cow['cluster'].unique())
    if unique_cluster>1:

        for cluster, data_cluster in data_cow.groupby('cluster'):
            adj_bred_final= data_cluster['breeding_date'].sort_values(ascending=True).iloc[0]
            data_cluster['adj_final_breeding']= adj_bred_final
            data_cluster['breed_event_id']= str(combined_cow[0])+'_'+str(combined_cow[1])+'_'+str(combined_cow[2])+'_'+str(cluster)
            temp.append(data_cluster)
    else :
        adj_bred_final= data_cow['breeding_date'].sort_values(ascending=True).iloc[0]
        data_cow['adj_final_breeding']= adj_bred_final
        data_cow['breed_event_id']= str(combined_cow[0])+'_'+str(combined_cow[1])+'_'+str(combined_cow[2])
        temp.append(data_cow)


        

In [15]:
import pandas as pd

# 1. Create a true datetime column. 
# This handles leap years, varying month lengths, and month/year rollovers automatically.
df['full_date'] = pd.to_datetime(df[['year', 'month', 'day']])

# 2. Sort by cow and date to ensure diff() calculates chronologically
df = df.sort_values(by=['combined_cow', 'full_date']).reset_index(drop=True)

# 3. Calculate difference in days strictly within each cow's history
df['date_diff'] = df.groupby('combined_cow')['full_date'].diff().dt.days

# 4. A new event is triggered if the gap is >= 2 days, or if it's the very first record (NaN)
df['check_new_event'] = (df['date_diff'] >= 2) | df['date_diff'].isna()

# 5. Create a global cluster ID for each cow using cumsum()
df['cluster'] = df.groupby('combined_cow')['check_new_event'].cumsum()

# 6. Assign the earliest breeding date per cluster
df['adj_final_breeding'] = df.groupby(['combined_cow', 'cluster'])['breeding_date'].transform('min')

# 7. Create the breed_event_id. 
# Since an event could start on March 31 and end on April 1, we should base 
# the ID on the year and month the cluster *started*.
cluster_starts = df.groupby(['combined_cow', 'cluster'])['full_date'].transform('min')
df['event_year'] = cluster_starts.dt.year.astype(str)
df['event_month'] = cluster_starts.dt.month.astype(str)

# Build the ID string: cow_year_month_cluster
df['breed_event_id'] = (
    df['combined_cow'].astype(str) + '_' + 
    df['event_year'] + '_' + 
    df['event_month'] + '_' + 
    df['cluster'].astype(str)
)

# Optional: Clean up helper columns
df = df.drop(columns=['event_year', 'event_month'])

In [9]:
data_cow= pd.concat(temp)

In [160]:
str(combined_cow[0])+'_'+str(combined_cow[1])+'_'+str(combined_cow[2])

'999999.0401430.0_2025_7'

In [21]:
data_final[data_final['combined_cow']=='10180.0401430.0'].sort_values('breeding_date', ascending=False)[['breeding_date', 'adj_final_breeding', 'day', 'month', 'year', 'breed_event_id']]

,breeding_date,adj_final_breeding,day,month,year,breed_event_id
428,2025-03-27,2025-03-26,27,3,2025,10180.0401430.0_2025_3
427,2025-03-26,2025-03-26,26,3,2025,10180.0401430.0_2025_3
426,2023-04-03,2023-04-02,3,4,2023,10180.0401430.0_2023_4
425,2023-04-02,2023-04-02,2,4,2023,10180.0401430.0_2023_4
422,2022-03-18,2022-03-17,18,3,2022,10180.0401430.0_2022_3
423,2022-03-18,2022-03-17,18,3,2022,10180.0401430.0_2022_3
424,2022-03-18,2022-03-17,18,3,2022,10180.0401430.0_2022_3
421,2022-03-17,2022-03-17,17,3,2022,10180.0401430.0_2022_3


In [16]:
temp_df=[]
for cow_id, data_id in df.groupby('combined_cow'):
    unique_breeding_event= data_id['adj_final_breeding'].unique()
    dict_event={value:index+1 for index,value in enumerate(unique_breeding_event)}
    data_id['ai_id']= data_id['adj_final_breeding'].apply(lambda x: dict_event[x])
    temp_df.append(data_id)
    

In [34]:
temp_df[120]

,Unnamed: 0,Date Collected,Date Received,CowID,Date Sent,Date Approved,ClientID,ProjectNo,Test,Result,...,year,month,day,month_day,date_diff,check_new_event,cluster,adj_final_breeding,breed_event_id,ai_id
578,7953,10/28/2022,10/29/2022 12:16,1025,10/28/2022 18:09,10/29/2022 17:06,401430,22100018,Pregnancy Test - Milk,Open,...,2022,9,22,09-22,NaN,False,0,2022-09-22,1025.0401430.0_2022_9,1
579,9193,12/9/2022,12/10/2022 15:19,1025,12/9/2022 18:11,12/10/2022 20:47,401430,22120008,Pregnancy Test - Milk,Open,...,2022,10,31,10-31,NaN,False,0,2022-10-31,1025.0401430.0_2022_10,2
580,1193,2/3/2023,2/4/2023 12:33,1025,2/3/2023 19:49,2/4/2023 18:00,401430,23020004,Pregnancy Test - Milk,Pregnant,...,2022,12,29,12-29,NaN,False,0,2022-12-29,1025.0401430.0_2022_12,3
581,5444,6/9/2023,6/10/2023 7:53,1025,6/9/2023 15:05,6/13/2023 13:02,401430,23060011,Pregnancy Test - Milk,Pregnant,...,2023,5,3,05-03,NaN,False,0,2023-05-03,1025.0401430.0_2023_5,4
582,6423,7/14/2023,7/15/2023 6:59,1025,7/14/2023 15:17,7/15/2023 11:02,401430,23070013,Pregnancy Test - Milk,Pregnant,...,2023,5,3,05-03,0.0,False,0,2023-05-03,1025.0401430.0_2023_5,4


In [17]:
data_final= pd.concat(temp_df)

In [29]:
data_final.columns

Index(['Unnamed: 0', 'Date Collected', 'Date Received', 'CowID', 'Date Sent',
       'Date Approved', 'ClientID', 'ProjectNo', 'Test', 'Result', 'S-N Value',
       'DIM', 'DSLB', 'combined_cow', 'date_collected', 'breeding_date',
       'Unnamed: 16', 'year', 'month', 'day', 'month_day', 'date_diff',
       'check_new_event', 'cluster', 'adj_final_breeding', 'breed_event_id',
       'ai_id'],
      dtype='object')

In [18]:
temp_sample=[]
for id, data_id in data_final.groupby(['combined_cow', 'ai_id']):
    unique_sample_event= data_id['date_collected'].unique()
    dict_event={value:index+1 for index,value in enumerate(unique_sample_event)}
    data_id['sample_id']= data_id['date_collected'].apply(lambda x: dict_event[x])
    temp_sample.append(data_id)

In [19]:

data_final_sample= pd.concat(temp_sample)

In [48]:
temp_sample[0].columns

Index(['Unnamed: 0', 'Date Collected', 'Date Received', 'CowID', 'Date Sent',
       'Date Approved', 'ClientID', 'ProjectNo', 'Test', 'Result', 'S-N Value',
       'DIM', 'DSLB', 'combined_cow', 'date_collected', 'breeding_date',
       'Unnamed: 16', 'year', 'month', 'day', 'month_day', 'date_diff',
       'check_new_event', 'cluster', 'adj_final_breeding', 'breed_event_id',
       'ai_id', 'sample_id'],
      dtype='object')

In [ ]:
temp_class=[]
for bred, data_bred in data_final_sample.groupby('breed_event_id'):
    max_event= max(data_bred['sample_id'])
    data_bred['sample_class']= data_bred['sample_id'].apply(lambda x: 'first' if x==1 else ('last' if x==max_event else 'intermediate'))
    temp_class.append(data_bred)
    

In [55]:
class_df= pd.concat(temp_class)

In [1]:
import pandas as pd 
import os 
import numpy as np 

class_df= pd.read_csv('data_with_sample_class.csv')

# condition time

In [68]:
class_df['Result'].value_counts()

Result
Pregnant    36213
Open         8134
Re-Check     3349
Name: count, dtype: int64

In [2]:
class_df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'Date Collected', 'Date Received',
       'CowID', 'Date Sent', 'Date Approved', 'ClientID', 'ProjectNo', 'Test',
       'Result', 'S-N Value', 'DIM', 'DSLB', 'combined_cow', 'date_collected',
       'breeding_date', 'Unnamed: 16', 'year', 'month', 'day', 'month_day',
       'date_diff', 'check_new_event', 'cluster', 'adj_final_breeding',
       'breed_event_id', 'ai_id', 'sample_id', 'sample_class'],
      dtype='object')

In [ ]:
temp_bred_data=[]
for bred, data_bred in class_df.groupby('breed_event_id'):
    if len(data_bred)==1:
        data_bred['final_resolved']= data_bred['Result'].iloc[0]
    
    elif:
        len(data_bred)==2:
        unique_samples = data_bred['Result'].unique()
        if len(unique_samples) == 1 and unique_samples[0] == 'Pregnant':
            data_bred['final_resolved'] = 'Pregnant'
       
    elif:
         len(data_bred)==2:
         samples= list(data_bred['Result'])
         if samples[0]=='Pregnant' & samples[1]=='Open':
            data_bred['final_resolved']=='loss'

    elif:

        len(data_bred)>2:
        samples= list(data_bred['Result'])

        if samples[0]=='Pregnant':
            samples_less= samples.pop(samples[0]):
            if 'Re-Check' in samples_less:
                last_sample= samples[-1]
                if last_sample=='Pregnant':
                    data_bred['final_resolved']='Pregant_with_rc'

                if last_sample=='Open':

                    data_bred['final_resolved']=='Preg_loss_with_rc'

    elif:
        len(data_bred)=2:
        samples= list(data_bred['Result'])

        if samples[0]=='Re-Check':
            samples_less= samples.pop(samples[0]):
           
                last_sample= samples[-1]
                if last_sample=='Open':
                    data_bred['final_resolved']='rc_open_2'

    elif:
        len(data_bred)>2:
        samples= list(data_bred['Result'])

        if samples[0]=='Re-Check':
            samples_less= samples.pop(samples[0]):
            if 'Pregnant' in samples_less:
                last_sample= samples[-1]
                if last_sample=='Open':
                    data_bred['final_resolved']='rc_preg_loss'

    elif:
        len(data_bred)>2:
        samples= list(data_bred['Result'])

        if samples[0]=='Re-Check':
            samples_less= samples.pop(samples[0]):
            if 'Re-Check' in samples_less:
                last_sample= samples[-1]
                if last_sample=='Open':
                    data_bred['final_resolved']='rc_open'

    else:
        len(data_bred)>2:
        samples= list(data_bred['Result'])

        if samples[0]=='Re-Check':
            samples_less= samples.pop(samples[0]):
            if 'Re-Check' in samples_less:
                last_sample= samples[-1]
                if last_sample=='Pregnant':
                    data_bred['final_resolved']='rc_preg'

        temp_bred_data.append(data_bred)

                        

            



['Pregnant']
['Pregnant']


In [4]:
import pandas as pd

temp_bred_data = []

for bred, data_bred in class_df.groupby('breed_event_id'):
    # Create a copy to avoid Pandas "SettingWithCopyWarning"
    data_bred = data_bred.copy()
    
    # Store length and convert results to a list for easier indexing
    n_samples = len(data_bred)
    samples = data_bred['Result'].tolist()
    
    # Default value in case a scenario isn't explicitly caught
    final_resolved = None 
    
    if n_samples == 1:
        final_resolved = None
        
    elif n_samples == 2:
        if samples[0] == 'Pregnant' and samples[1] == 'Pregnant':
            final_resolved = 'Pregnant'
        elif samples[0] == 'Pregnant' and samples[1] == 'Open':
            final_resolved = 'loss'
        elif samples[0] == 'Re-Check' and samples[1] == 'Open':
            final_resolved = 'rc_open_2'
            
    elif n_samples > 2:
        first_sample = samples[0]
        last_sample = samples[-1]
        samples_less = samples[1:-1] 
        
        if first_sample == 'Pregnant':
            if 'Re-Check' in samples_less:
                if last_sample == 'Pregnant':
                    final_resolved = 'Pregant_with_rc'
                elif last_sample == 'Open':
                    final_resolved = 'Preg_loss_with_rc'
                    
        elif first_sample == 'Re-Check':
            if 'Pregnant' in samples_less:
                if last_sample == 'Open':
                    final_resolved = 'rc_preg_loss'
            elif 'Re-Check' in samples_less:
                if last_sample == 'Open':
                    final_resolved = 'rc_open'
                elif last_sample == 'Pregnant':
                    final_resolved = 'rc_preg'
    
    # Assign the calculated outcome to the whole group
    data_bred['final_resolved'] = final_resolved
    
    # Append MUST be outside the if/elif blocks so every group is saved
    temp_bred_data.append(data_bred)

# Recombine everything into a single dataframe at the end
final_df = pd.concat(temp_bred_data, ignore_index=True)

In [7]:
final_df[final_df['combined_cow']=='10069.0401430.0'].sort_values('breeding_date', ascending=False)[['breeding_date', 'adj_final_breeding', 'day', 'month', 'year', 'breed_event_id', 'Result', 'final_resolved', 'DSLB', 'ai_id']]

,breeding_date,adj_final_breeding,day,month,year,breed_event_id,Result,final_resolved,DSLB,ai_id
164,2023-04-01,2023-04-01,1,4,2023,10069.0401430.0_2023_4,Re-Check,None,76,3
163,2023-03-31,2023-03-31,31,3,2023,10069.0401430.0_2023_3,Pregnant,None,42,2
160,2022-03-16,2022-03-16,16,3,2022,10069.0401430.0_2022_3,Pregnant,Pregant_with_rc,30,1
161,2022-03-16,2022-03-16,16,3,2022,10069.0401430.0_2022_3,Re-Check,Pregant_with_rc,72,1
162,2022-03-16,2022-03-16,16,3,2022,10069.0401430.0_2022_3,Pregnant,Pregant_with_rc,79,1


In [2]:
import pandas as pd 
import os 
final_df= pd.read_csv('2_27_data.csv')

In [27]:
for cow_id, data_id in final_df.groupby('combined_cow'):
    unique_dates = sorted(data_id['adj_final_breeding'].unique())
    
    # Calculate gaps
    gap_dates = {unique_dates[i]: (unique_dates[i+1] - unique_dates[i]).days 
                 for i in range(len(unique_dates) - 1)}
    
    # Filter for gaps < 300 days
    selected_gap = {date: gap for date, gap in gap_dates.items() if gap < 300}

    for adj_date in selected_gap.keys():
        # Get data for this specific date
        data_adj_date = data_id[data_id['adj_final_breeding'] == adj_date].copy()
        
        if len(data_adj_date) > 1:
            results_list = data_adj_date['Result'].tolist()
            last_sample = results_list[-1]

            # 1. Skip if the last status of the day was 'Open'
            if last_sample == 'Open':
                continue
            
            # 2. Check for Pregnancy Loss / Re-breeding
            if "Pregnant" in results_list:
                # Use .loc to avoid the "SettingWithCopy" warning
                final_df.loc[data_adj_date.index, 'final_resolved'] = 'preg_loss_rebred'

            # 3. Check if EVERYTHING that day was a 'Re-Check'
            if all(res == 'Re-Check' for res in results_list):
                final_df.loc[data_adj_date.index, 'final_resolved'] = 'rc_rebred'

In [29]:
final_df.to_csv('3_2_data.csv')

In [23]:
data_adj_date[['Result', 'final_resolved', 'ai_id', 'adj_final_breeding', 'sample_id']]

,Result,final_resolved,ai_id,adj_final_breeding,sample_id
46673,Pregnant,loss,3,2023-04-25,1
46674,Open,loss,3,2023-04-25,2


In [24]:
data_adj_date['Result'].value_counts()

Result
Pregnant    1
Open        1
Name: count, dtype: int64

In [38]:
for cow, data_cow in df.groupby('combined_cow'):
    test_str = list(data_cow['Result'].values)
    
    # First, make sure the cow actually has at least 2 results to avoid an IndexError
    if len(test_str) >= 2:
        # Check if both the first and second results are 'Re-Check'
        if test_str[0] == 'Re-Check' and test_str[1] == 'Re-Check':
            print(cow)

10076.0401430.0
10669.0401430.0
10695.0401430.0
10705.0401430.0
10727.0401430.0
10756.0401430.0
10799.0401430.0
10841.0401430.0
11036.0401430.0
11308.0401430.0
11362.0401430.0
11457.0401430.0
11579.0401430.0
11693.0401430.0
12053.0401430.0
12410.0401430.0
12464.0401430.0
12572.0401430.0
12608.0401430.0
1267.0401430.0
12806.0401430.0
12929.0401430.0
13345.0401430.0
13400.0401430.0
13606.0401430.0
13923.0401430.0
13968.0401430.0
14046.0401430.0
14059.0401430.0
14168.0401430.0
14497.0401430.0
14535.0401430.0
14538.0401430.0
14650.0401430.0
14891.0401430.0
15000.0401430.0
15020.0401430.0
15049.0401430.0
15420.0401430.0
15627.0401430.0
15846.0401430.0
15898.0401430.0
15956.0401430.0
15988.0401430.0
16556.0401430.0
1667.0401430.0
16887.0401430.0
17302.0401430.0
17346.0401430.0
19110.0401430.0
19139.0401430.0
19781.0401430.0
2237.0401430.0
24730.0392159.0
2514.0401430.0
2785.0401430.0
3223.0392159.0
3302.0401430.0
3480.0392159.0
3860.0392159.0
3890.0392159.0
3966.0401430.0
3995.0401430.0
4136

# ordering dataset

In [7]:
df = df.sort_values(
    ["combined_cow", "breeding_date", "date_collected"]
)


In [8]:
df = df.sort_values(
    ["combined_cow", "breeding_date", "date_collected"]
)


In [9]:
df["breeding_event_id"] = (
    df.groupby("combined_cow")["breeding_date"]
      .rank(method="dense")
      .astype(int)
)


IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer